# SafeMask + CS 136 Preprocessing on Google Colab

Steps in this notebook:
1. Clone the repo and install dependencies
2. Mount your Google Drive (where ACDC images live)
3. Run a small CS 136 preprocessing sample (optional)
4. Train SafeMask with CS 136 preprocessing on, ~1 to 1.5 hours on a T4

**Before running:** make sure you have selected a GPU runtime (Runtime > Change runtime type > T4 GPU) and uploaded the ACDC `rgb_anon_trainvaltest` and `gt_trainval` folders to your Google Drive.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/jenilkathrotia/SafeMask.git
%cd SafeMask
!pip install -q -r requirements.txt
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless

## 2. Mount Google Drive
Adjust `DRIVE_DATA_ROOT` to the folder where you uploaded ACDC.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CHANGE THIS to wherever you uploaded the ACDC folders on Drive
DRIVE_DATA_ROOT = '/content/drive/MyDrive/ACDC'

RGB  = f'{DRIVE_DATA_ROOT}/rgb_anon_trainvaltest/rgb_anon'
MASK = f'{DRIVE_DATA_ROOT}/gt_trainval/gt'

!ls $RGB && ls $MASK

## 3. Optional: run CS 136 preprocessing on a small sample
Skip this cell if you only care about training. Takes ~5-10 minutes for 20 images.

In [ ]:
!python cs136_preprocessing/Part1_Preprocessing/01_Gaussian_Filter/gaussian_filter.py \
    --input-dir $RGB --split train --per-condition 5

!python cs136_preprocessing/Part3_Robustness/02_Pipeline_Comparison/compare_pipelines.py \
    --input-dir $RGB --split train --per-condition 5

## 4. Update SafeMask config to point at Drive paths
ACDC paths use a glob pattern that matches all four weather folders at once.

In [ ]:
import yaml
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['train_image_dir'] = f'{RGB}'  # SegmentationDataset will recurse
cfg['dataset']['val_image_dir']   = f'{RGB}'
cfg['dataset']['train_mask_dir']  = f'{MASK}'
cfg['dataset']['val_mask_dir']    = f'{MASK}'

# CS 136 preprocessing is on by default. Leave it on.
print(cfg['cs136_preprocessing'])

with open('configs/config.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('config updated')

## 5. Train SafeMask
First run downloads the ResNet weights, then training begins. ~1 to 1.5 hours on a T4 GPU.

In [ ]:
!python scripts/train.py --config configs/config.yaml

## 6. Inference on one image (optional)

In [ ]:
import glob
test_img = glob.glob(f'{RGB}/fog/test/**/*.png', recursive=True)[0]
print('Using:', test_img)
!python scripts/infer.py --image "$test_img" --output outputs/visualizations/colab_demo.png
from IPython.display import Image
Image('outputs/visualizations/colab_demo.png')